# EP1 Machine Learning — Spotify Tracks
## Inteligencia musical y predicción de popularidad

**Asignatura:** MLY1101 Machine Learning | **Institución:** Duoc UC | **Metodología:** CRISP-DM

> **Objetivo de esta EP1:** documentar y reproducir las etapas de comprensión del problema, identificación de fuentes, preparación de datos, EDA completo, calidad de datos, sesgos, ética y privacidad.

> **Target principal:** `popularity` (regresión, 0–100).

> Este notebook no entrena el modelo final. Deja los datos y el pipeline preparados para la etapa de modelamiento (EP2).


## 0. Imports y configuración

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats as scipy_stats
from sklearn.model_selection import GroupShuffleSplit
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline

pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', lambda x: f'{x:,.4f}')
plt.rcParams['figure.dpi'] = 120
sns.set_theme(style='whitegrid', palette='muted')

# Rutas robustas — funciona tanto desde notebooks/ como desde la raíz
cwd = Path.cwd()
if cwd.name == 'notebooks':
    PROJECT_ROOT = cwd.parent
elif (cwd / 'data' / 'raw' / 'Spotify_Tracks_Dataset.csv').exists():
    PROJECT_ROOT = cwd
else:
    PROJECT_ROOT = cwd.parent

DATA_PATH      = PROJECT_ROOT / 'data' / 'raw' / 'Spotify_Tracks_Dataset.csv'
PROCESSED_DIR  = PROJECT_ROOT / 'data' / 'processed'
IMAGES_DIR     = PROJECT_ROOT / 'images'
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
IMAGES_DIR.mkdir(parents=True, exist_ok=True)

print('Proyecto:', PROJECT_ROOT)
print('Dataset:', DATA_PATH)


## 1. Metodología CRISP-DM

El proyecto sigue CRISP-DM. Esta EP1 cubre las **fases 1–3**:

| Fase | Descripción | Estado EP1 |
|------|-------------|------------|
| 1. Business Understanding | Problema, objetivos, KPIs | ✅ Completo |
| 2. Data Understanding | EDA, calidad, relaciones | ✅ Completo |
| 3. Data Preparation | Limpieza, transformaciones, pipeline | ✅ Completo |
| 4. Modeling | Entrenamiento de modelos | 🔄 EP2 |
| 5. Evaluation | Métricas y comparación | 🔄 EP2 |
| 6. Deployment | Scoring en producción | 🔄 EP3 |


## 2. Problema de negocio, objetivos y KPIs

### Problema de negocio
Un equipo de inteligencia musical necesita determinar si los **atributos medibles de audio** de una canción permiten anticipar su nivel de popularidad para apoyar decisiones de curaduría editorial y promoción algorítmica.

### Objetivo general
Construir una base analítica reproducible para desarrollar un modelo de regresión que estime `popularity` a partir de features musicales.

### Objetivos específicos
- **Analítico:** identificar qué atributos se asocian con mayor popularidad.
- **ML:** entrenar un modelo de regresión con `popularity` como target.

### KPIs
| KPI | Umbral de referencia |
|-----|---------------------|
| Cobertura de datos (sin nulos en features) | ≥ 99% |
| Contaminación train/test (track_id compartidos) | 0% |
| MAE (fase futura) | < 10 puntos |
| RMSE (fase futura) | < 15 puntos |
| R² (fase futura) | > 0.20 |


## 3. Fuentes de datos y herramientas colaborativas

**Fuente:** `Spotify_Tracks_Dataset.csv` — Kaggle (*Spotify Tracks Dataset*, MaharshiPandya).

**Limitación:** sin fecha de extracción ni versión de API. La `popularity` puede estar desactualizada.

| Herramienta | Uso |
|-------------|-----|
| Python / pandas / numpy | Manipulación de datos |
| matplotlib / seaborn | Visualización |
| scikit-learn | Pipeline ML |
| Jupyter Notebook | Documentación reproducible |
| Git / GitHub | Control de versiones grupal |
| Google Colab | Ejecución compartida |


## 4. Carga y estructura del dataset

In [ ]:
df_raw = pd.read_csv(DATA_PATH)
print(f'Dimensiones originales: {df_raw.shape[0]:,} filas x {df_raw.shape[1]} columnas')
df_raw.head(3)


## 5. Diccionario y clasificación analítica de variables

| Variable | Tipo | Rol | Descripción |
|----------|------|-----|-------------|
| `track_id` | Texto | Identificador | ID único de Spotify — excluir del modelo |
| `artists` | Texto | Metadato | Nombre(s) del artista |
| `album_name` | Texto | Metadato | Nombre del álbum |
| `track_name` | Texto | Metadato | Nombre de la canción |
| `popularity` | Numérica (int) | **TARGET** | Popularidad 0–100 |
| `duration_ms` | Numérica | Feature | Duración en ms |
| `explicit` | Booleana | Feature | Contenido explícito |
| `danceability` | Float 0–1 | Feature | Aptitud para bailar |
| `energy` | Float 0–1 | Feature | Intensidad percibida |
| `key` | Categórica | Feature | Tonalidad (0=C…11=B) |
| `loudness` | Float (dB) | Feature | Volumen promedio |
| `mode` | Categórica | Feature | Mayor(1)/Menor(0) |
| `speechiness` | Float 0–1 | Feature | Presencia de voz hablada |
| `acousticness` | Float 0–1 | Feature | Nivel acústico |
| `instrumentalness` | Float 0–1 | Feature | Nivel instrumental |
| `liveness` | Float 0–1 | Feature | Presencia de audiencia en vivo |
| `valence` | Float 0–1 | Feature | Positividad musical |
| `tempo` | Float (BPM) | Feature | Velocidad |
| `time_signature` | Categórica | Feature | Compás musical |
| `track_genre` | Categórica | Feature | Género (114 categorías) |

> **Nota:** `key`, `mode` y `time_signature` son **códigos musicales**, no magnitudes. Se tratarán como categóricas.


In [ ]:
# Inspección de estructura
estructura = pd.DataFrame({
    'variable': df_raw.columns,
    'dtype':    df_raw.dtypes.astype(str).values,
    'no_nulos': df_raw.notna().sum().values,
    'nulos':    df_raw.isna().sum().values,
    'unicos':   df_raw.nunique(dropna=True).values
})
display(estructura)


## 6. Limpieza y transformaciones justificadas

### Decisiones tomadas:
1. **Eliminar `Unnamed: 0`:** índice artificial, no aporta información.
2. **Eliminar la fila con NaN en metadatos de texto:** solo 1 fila afectada (0.001% de pérdida).
3. **Crear `duration_min`:** más interpretable que milisegundos.
4. **Conservar outliers:** todos los valores extremos son musicalmente válidos.
5. **No imputar:** no hay NaN en features de audio ni en el target.


In [ ]:
df = df_raw.drop(columns=['Unnamed: 0']).copy()

# Verificar NaN
missing = df.isna().sum()
print('Celdas nulas por variable:')
print(missing[missing > 0])
print(f'\nFila(s) afectada(s):')
display(df[df.isna().any(axis=1)])

# Eliminar la fila con NaN en metadatos
before = len(df)
df_clean = df.dropna(subset=['artists', 'album_name', 'track_name']).copy()
after = len(df_clean)

# Feature interpretable
df_clean['duration_min'] = df_clean['duration_ms'] / 60000

print(f'\nFilas antes:  {before:,}')
print(f'Filas después: {after:,}')
print(f'Pérdida: {before-after} fila(s) ({(before-after)/before*100:.4f}%)')

# Guardar versión limpia
df_clean.to_csv(PROCESSED_DIR / 'spotify_clean.csv', index=False)
print('Dataset limpio guardado.')


In [ ]:
# Clasificación de variables para el análisis
numeric_audio = [
    'duration_ms', 'danceability', 'energy', 'loudness', 'speechiness',
    'acousticness', 'instrumentalness', 'liveness', 'valence', 'tempo'
]
categorical_features = ['explicit', 'key', 'mode', 'time_signature', 'track_genre']
text_metadata        = ['track_id', 'artists', 'album_name', 'track_name']
numeric_desc = ['popularity', 'duration_ms', 'duration_min'] + numeric_audio[1:]

print('Numéricas predictoras:', numeric_audio)
print('Categóricas predictoras:', categorical_features)
print('Metadatos (excluir del modelo):', text_metadata)


## 7. Estadística descriptiva completa

Se calculan media, mediana, desviación estándar, percentiles 25/50/75/90/95/99, rango, skewness y curtosis para cada variable numérica.


In [ ]:
# Estadística descriptiva extendida con skewness y curtosis
rows = []
for col in numeric_desc:
    s = df_clean[col].dropna()
    q25, q50, q75, q90, q95, q99 = s.quantile([0.25, 0.50, 0.75, 0.90, 0.95, 0.99])
    rows.append({
        'variable':   col,
        'count':      int(len(s)),
        'mean':       s.mean(),
        'median':     s.median(),
        'std':        s.std(),
        'min':        s.min(),
        'P25':        q25,
        'P75':        q75,
        'P90':        q90,
        'P95':        q95,
        'P99':        q99,
        'max':        s.max(),
        'range':      s.max() - s.min(),
        'skewness':   s.skew(),
        'kurtosis':   s.kurtosis()
    })

stats_df = pd.DataFrame(rows).set_index('variable')
display(stats_df.round(4))
stats_df.to_csv(PROCESSED_DIR / 'descriptive_statistics.csv')
print('Estadísticas guardadas.')


### Interpretación de hallazgos clave

- **`popularity`:** media 33.24, mediana 35.0 — distribución casi simétrica pero bimodal (14.1% con valor 0). Solo 4.8% supera 70.
- **`duration_ms`:** skewness 11.20 — distribución extremadamente asimétrica. La canción más larga dura 87.3 minutos (outlier extremo).
- **`speechiness`:** skewness 4.65, curtosis 28.82 — concentrada en valores muy bajos. La mayoría son música, no spoken word.
- **`instrumentalness`:** mediana exactamente 0 — la mayoría de canciones tienen voz. Distribución bimodal.
- **`valence`:** curtosis -1.03 — distribución casi uniforme, sin pico dominante. Dataset emocionalmente diverso.
- **`energy` y `loudness`:** ambas asimétricas a la izquierda; canciones muy apagadas son menos comunes.


## 8. Histogramas y distribuciones

In [ ]:
fig, axes = plt.subplots(4, 3, figsize=(15, 14))
axes = axes.flatten()

for i, col in enumerate(numeric_desc):
    ax = axes[i]
    data = df_clean[col].dropna()
    ax.hist(data, bins=40, edgecolor='black', alpha=0.7, color='steelblue')
    ax.set_title(f'Distribución: {col}', fontsize=11)
    ax.set_xlabel(col, fontsize=9)
    ax.set_ylabel('Frecuencia', fontsize=9)
    # Añadir líneas de media y mediana
    ax.axvline(data.mean(),   color='red',    linestyle='--', linewidth=1.5, label=f'Media: {data.mean():.2f}')
    ax.axvline(data.median(), color='orange', linestyle='-',  linewidth=1.5, label=f'Mediana: {data.median():.2f}')
    ax.legend(fontsize=7)

# Ocultar subplots vacíos
for j in range(len(numeric_desc), len(axes)):
    axes[j].set_visible(False)

plt.suptitle('Distribuciones de variables numéricas', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig(IMAGES_DIR / '01_histogramas_numericas.png', dpi=150, bbox_inches='tight')
plt.show()
print('Gráfico guardado: 01_histogramas_numericas.png')


## 9. Variables categóricas: frecuencias y proporciones

Se analiza la distribución de `explicit`, `key`, `mode`, `time_signature` y `track_genre`.


In [ ]:
def frequency_table(data, variable):
    counts = data[variable].value_counts(dropna=False)
    pct    = data[variable].value_counts(dropna=False, normalize=True) * 100
    return pd.DataFrame({'frecuencia': counts, 'proporcion_pct': pct.round(2)})

for variable in ['explicit', 'key', 'mode', 'time_signature']:
    print(f'\n── {variable} ──')
    display(frequency_table(df_clean, variable))

print(f'\n── track_genre: {df_clean["track_genre"].nunique()} géneros únicos (mostrando top 20) ──')
display(frequency_table(df_clean, 'track_genre').head(20))


In [ ]:
# Gráficos de variables categóricas
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

for ax, variable in zip(axes.flatten(), ['explicit', 'key', 'mode', 'time_signature']):
    counts = df_clean[variable].value_counts().sort_index()
    ax.bar(counts.index.astype(str), counts.values, color='steelblue', edgecolor='black')
    ax.set_title(f'Frecuencia de {variable}', fontsize=12, fontweight='bold')
    ax.set_xlabel(variable, fontsize=10)
    ax.set_ylabel('Frecuencia', fontsize=10)
    for bar, val in zip(ax.patches, counts.values):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 200,
                f'{val:,}', ha='center', va='bottom', fontsize=8)

plt.suptitle('Distribución de variables categóricas', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(IMAGES_DIR / '02_variables_categoricas.png', dpi=150, bbox_inches='tight')
plt.show()


## 10. Calidad de datos: checks de dominio y duplicados

Se verifica que todos los valores estén dentro de los rangos documentados por la API de Spotify.


In [ ]:
# Verificaciones de dominio
checks = {
    'popularity fuera de 0-100':      ((df_clean['popularity'] < 0) | (df_clean['popularity'] > 100)).sum(),
    'danceability fuera de 0-1':      ((df_clean['danceability'] < 0) | (df_clean['danceability'] > 1)).sum(),
    'energy fuera de 0-1':            ((df_clean['energy'] < 0) | (df_clean['energy'] > 1)).sum(),
    'speechiness fuera de 0-1':       ((df_clean['speechiness'] < 0) | (df_clean['speechiness'] > 1)).sum(),
    'acousticness fuera de 0-1':      ((df_clean['acousticness'] < 0) | (df_clean['acousticness'] > 1)).sum(),
    'instrumentalness fuera de 0-1':  ((df_clean['instrumentalness'] < 0) | (df_clean['instrumentalness'] > 1)).sum(),
    'liveness fuera de 0-1':          ((df_clean['liveness'] < 0) | (df_clean['liveness'] > 1)).sum(),
    'valence fuera de 0-1':           ((df_clean['valence'] < 0) | (df_clean['valence'] > 1)).sum(),
    'key fuera de 0-11':              ((df_clean['key'] < 0) | (df_clean['key'] > 11)).sum(),
    'mode fuera de {0,1}':            (~df_clean['mode'].isin([0, 1])).sum(),
    'loudness > 0 dB (inusual)':      (df_clean['loudness'] > 0).sum(),
    'duration_ms <= 0':               (df_clean['duration_ms'] <= 0).sum(),
    'tempo == 0 (inusual)':           (df_clean['tempo'] == 0).sum(),
    'time_signature == 0 (inusual)':  (df_clean['time_signature'] == 0).sum(),
}

checks_df = pd.DataFrame(list(checks.items()), columns=['Regla', 'Cantidad'])
checks_df['Estado'] = checks_df['Cantidad'].apply(lambda x: '✅ OK' if x == 0 else '⚠️ Revisar')
display(checks_df)

# Duplicados
print(f'\nDuplicados exactos: {df_clean.duplicated().sum()}')
print(f'track_id únicos: {df_clean["track_id"].nunique():,}')
track_counts = df_clean['track_id'].value_counts()
print(f'IDs que aparecen en más de un género: {(track_counts > 1).sum():,}')
print(f'Máximo apariciones del mismo track_id: {track_counts.max()}')
print(f'\n⚠️ Riesgo de leakage si no se controla el split por track_id.')


## 11. Outliers mediante IQR

El criterio IQR se usa para **detectar**, no para eliminar automáticamente.
Cada caso se evalúa individualmente antes de tomar una decisión.


In [ ]:
outlier_rows = []
for col in numeric_audio + ['popularity']:
    s = df_clean[col].dropna()
    q1, q3 = s.quantile([0.25, 0.75])
    iqr = q3 - q1
    low  = q1 - 1.5 * iqr
    high = q3 + 1.5 * iqr
    mask = (s < low) | (s > high)
    outlier_rows.append({
        'variable':      col,
        'Q1':            q1,
        'Q3':            q3,
        'limite_inf':    low,
        'limite_sup':    high,
        'n_outliers':    int(mask.sum()),
        'pct_outliers':  round(mask.mean() * 100, 2),
        'decision':      'Conservar'
    })

outliers_df = pd.DataFrame(outlier_rows).sort_values('pct_outliers', ascending=False)
display(outliers_df.round(3))
outliers_df.to_csv(PROCESSED_DIR / 'outliers_iqr_summary.csv', index=False)
print('\nResumen de outliers guardado.')


### Justificación de las decisiones:

- **`instrumentalness` (22.1%):** distribución bimodal legítima — canciones con y sin voz son categorías musicales reales.
- **`speechiness` (11.6%):** rap, spoken word y podcasts tienen valores altos. Son música real.
- **`liveness` (7.6%):** grabaciones en vivo existen y son válidas.
- **`loudness` (5.4%):** el rango de dB es válido musicalmente; algunos valores positivos corresponden a audio con compresión agresiva.
- **`duration_ms` (4.9%):** la canción de 87.3 minutos es un caso extremo real (compilado/meditación). No se elimina sin regla de negocio.
- **`popularity` (0.002%):** los 2 outliers son canciones con popularity=100, el valor máximo posible. Son correctos.

**Conclusión:** no se elimina ningún outlier. Todos representan valores musicalmente válidos.


## 12. Correlaciones y relaciones entre variables

In [ ]:
corr_cols = numeric_audio + ['popularity']
corr = df_clean[corr_cols].corr()

# Heatmap
fig, ax = plt.subplots(figsize=(12, 9))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='RdBu_r', center=0,
            vmin=-1, vmax=1, ax=ax, annot_kws={'size': 8})
ax.set_title('Matriz de correlación de variables numéricas', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(IMAGES_DIR / '03_matriz_correlacion.png', dpi=150, bbox_inches='tight')
plt.show()

# Correlaciones con popularity
corr_pop = corr['popularity'].drop('popularity').sort_values(key=lambda s: s.abs(), ascending=False)
print('=== Correlaciones con popularity (ordenadas por valor absoluto) ===')
display(corr_pop.to_frame('correlacion_con_popularity').round(4))
print('\nNota: Todas las correlaciones son menores a 0.10 en valor absoluto.')
print('Los features de audio solos no explican bien la popularidad.')


## 13. Comparaciones categóricas: popularidad por grupos

In [ ]:
# Por género
genre_stats = (
    df_clean.groupby('track_genre')
    .agg(
        n               = ('track_id', 'size'),
        popularity_mean = ('popularity', 'mean'),
        popularity_med  = ('popularity', 'median'),
        popularity_std  = ('popularity', 'std'),
        dance_mean      = ('danceability', 'mean'),
        energy_mean     = ('energy', 'mean')
    )
    .sort_values('popularity_mean', ascending=False)
    .round(2)
)

print('=== TOP 10 géneros por popularidad media ===')
display(genre_stats.head(10))
print('\n=== BOTTOM 10 géneros por popularidad media ===')
display(genre_stats.tail(10))

genre_stats.to_csv(PROCESSED_DIR / 'genre_popularity_stats.csv')


In [ ]:
# Gráfico Top 15 y Bottom 15
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 8))

top15  = genre_stats.head(15).sort_values('popularity_mean')
bot15  = genre_stats.tail(15).sort_values('popularity_mean', ascending=False)

ax1.barh(top15.index, top15['popularity_mean'], color='steelblue', edgecolor='black')
ax1.set_title('Top 15 géneros\n(mayor popularidad media)', fontsize=12, fontweight='bold')
ax1.set_xlabel('Popularidad media')
ax1.axvline(df_clean['popularity'].mean(), color='red', linestyle='--', label=f'Media global ({df_clean["popularity"].mean():.1f})')
ax1.legend(fontsize=9)

ax2.barh(bot15.index, bot15['popularity_mean'], color='salmon', edgecolor='black')
ax2.set_title('Bottom 15 géneros\n(menor popularidad media)', fontsize=12, fontweight='bold')
ax2.set_xlabel('Popularidad media')
ax2.axvline(df_clean['popularity'].mean(), color='red', linestyle='--', label=f'Media global')
ax2.legend(fontsize=9)

plt.suptitle('Popularidad media por género', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(IMAGES_DIR / '04_popularidad_por_genero.png', dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
# Por explicit
explicit_stats = df_clean.groupby('explicit')['popularity'].agg(['count','mean','median','std']).round(2)
print('=== Popularidad según contenido explícito ===')
display(explicit_stats)

fig, ax = plt.subplots(figsize=(7, 5))
groups_data = [df_clean.loc[df_clean['explicit'] == False, 'popularity'].values,
               df_clean.loc[df_clean['explicit'] == True,  'popularity'].values]
bp = ax.boxplot(groups_data, labels=['No explícita', 'Explícita'], patch_artist=True,
                showfliers=False)
bp['boxes'][0].set_facecolor('steelblue')
bp['boxes'][1].set_facecolor('salmon')
ax.set_title('Popularidad según contenido explícito', fontsize=12, fontweight='bold')
ax.set_ylabel('Popularidad')
plt.tight_layout()
plt.savefig(IMAGES_DIR / '05_popularidad_explicit.png', dpi=150, bbox_inches='tight')
plt.show()
print('Diferencia de medias: {:.2f} puntos (no implica causalidad)'.format(
    explicit_stats.loc[True,'mean'] - explicit_stats.loc[False,'mean']))


## 14. Análisis adicional: perfil de canciones populares

Se comparan los features de audio entre canciones con `popularity ≥ 70` y el resto.


In [ ]:
pop_alta  = df_clean[df_clean['popularity'] >= 70]
pop_baja  = df_clean[df_clean['popularity'] < 70]

print(f'Canciones con popularity >= 70: {len(pop_alta):,} ({len(pop_alta)/len(df_clean)*100:.1f}%)')
print(f'Resto: {len(pop_baja):,}')

comparacion = pd.DataFrame({
    'Popular (≥70)':   pop_alta[numeric_audio].mean(),
    'Resto (<70)':     pop_baja[numeric_audio].mean(),
}).round(4)
comparacion['Diferencia'] = (comparacion['Popular (≥70)'] - comparacion['Resto (<70)']).round(4)
display(comparacion)
print('\nObservación: canciones muy populares tienen mayor danceability, mayor loudness y MENOR instrumentalness.')


## 15. Preparación para Machine Learning

### Decisiones de diseño:
- **Target:** `popularity` (regresión numérica).
- **Excluir:** `track_id` (identificador), `artists`, `album_name`, `track_name` (alta cardinalidad), `duration_min` (derivada).
- **Numéricas:** StandardScaler (más robusto a outliers que MinMaxScaler).
- **Categóricas:** OneHotEncoder (evita orden implícito de Label Encoding).
- **Split:** GroupShuffleSplit por `track_id` para evitar data leakage.


In [ ]:
model_numeric      = [
    'duration_ms', 'danceability', 'energy', 'loudness', 'speechiness',
    'acousticness', 'instrumentalness', 'liveness', 'valence', 'tempo'
]
model_categorical  = ['explicit', 'key', 'mode', 'time_signature', 'track_genre']

X      = df_clean[model_numeric + model_categorical].copy()
y      = df_clean['popularity'].copy()
groups = df_clean['track_id'].copy()

# Split sin leakage
gss = GroupShuffleSplit(n_splits=1, test_size=0.20, random_state=42)
train_idx, test_idx = next(gss.split(X, y, groups=groups))

X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

# Verificación de cero contaminación
train_ids = set(groups.iloc[train_idx])
test_ids  = set(groups.iloc[test_idx])
overlap   = train_ids & test_ids

print(f'Train: {X_train.shape}')
print(f'Test:  {X_test.shape}')
print(f'track_id compartidos entre train y test: {len(overlap)}')
assert len(overlap) == 0, 'ERROR: existe fuga de track_id'
print('✅ Verificación de cero leakage: OK')


In [ ]:
# Pipeline reproducible
numeric_transformer     = Pipeline([('scaler', StandardScaler())])
categorical_transformer = Pipeline([('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))])

preprocessor = ColumnTransformer(transformers=[
    ('num', numeric_transformer,     model_numeric),
    ('cat', categorical_transformer, model_categorical),
])

# Ajuste SOLO con datos de entrenamiento
preprocessor.fit(X_train)

X_train_prep = preprocessor.transform(X_train)
X_test_prep  = preprocessor.transform(X_test)

print(f'Matriz train transformada: {X_train_prep.shape}')
print(f'Matriz test transformada:  {X_test_prep.shape}')
print('\n✅ Pipeline preparado correctamente. StandardScaler ajustado SOLO con train.')
print('✅ El test nunca fue visto durante el ajuste del preprocesador.')


## 16. Sesgos, ética y privacidad

### Sesgos identificados

| Sesgo | Tipo | Descripción |
|-------|------|-------------|
| **Muestreo artificial balanceado** | Observado | 1,000 canciones por género exactamente — no refleja el mercado real |
| **Feedback loop** | Potencial | Popularidad predicha → más promoción → más reproducciones → ciclo autoreforzante |
| **Sesgo de exposición por género** | Potencial | Géneros no anglosajones tienen menor popularidad, no necesariamente por calidad |
| **Sesgo por artista** | Observado | Artistas con más discografía tienen más canciones y mayor probabilidad de éxito |
| **Temporalidad** | Metodológico | Sin fecha de extracción, la popularidad puede estar desactualizada |

### Privacidad
El CSV no contiene datos de usuarios. El riesgo de privacidad individual es **bajo**.
Si se incorporaran historiales de reproducción: minimización, pseudonimización, base legal (GDPR/Ley 19.628 Chile).

### Medidas de mitigación
- Evaluar MAE/RMSE **por género** (equidad algorítmica).
- Usar GroupShuffleSplit (implementado).
- Documentar limitaciones para usuarios finales.
- Mantener supervisión humana en decisiones editoriales.
- No comunicar predicciones como medida de calidad artística.


## 17. Conclusiones de la EP1

1. **Calidad estructural alta:** 3 celdas nulas de 2.394 millones. El dataset está bien construido.
2. **Correlaciones débiles (< 0.10):** los features de audio solos no predicen bien la popularidad. El R² del modelo será bajo sin features adicionales.
3. **`track_genre` es el predictor más potente disponible** (diferencia de 57 puntos entre géneros más y menos populares).
4. **El 14.1% de canciones tiene popularity = 0** — bimodalidad que el modelo debe manejar.
5. **Duplicados por track_id resueltos** con GroupShuffleSplit (cero contaminación verificada).
6. **Pipeline reproducible listo** para recibir cualquier modelo de regresión o clasificación en EP2.
7. **Los outliers son valores válidos** — no se eliminan sin regla de negocio.

### Indicadores de rúbrica cubiertos:
- **IE1 (10%):** fuentes y herramientas → secciones 3 y 4 ✅
- **IE2 (30%):** manipulación y preparación Python → secciones 6, 10, 15 ✅
- **IE3 (40%):** EDA completo y calidad → secciones 7–14 ✅
- **IE4 (20%):** sesgos, ética y privacidad → sección 16 ✅

El informe técnico completo está en `README.md`.
